# 08 — Context Managers and Resource Safety

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/08', {1: '9ef0a6527b9e0c62', 2: 'f4b50942c22514d2', 3: '7e621bfff96363c8', 4: '05eca1d24a5c04d3', 5: '230a51634cebe87c', 6: '5eea4ff0eb88d5b7', 7: '2c830bca3a4996e6', 8: '7fa753aff098f9c9', 9: '0974d5aa16d6f7b9'}, practice='intermediate_02_iteration_and_abstractions')

## 1. with

- `with resource as name:` sets something up, runs the block, then always cleans up, even if the block raises.
- Files are the classic case: `with open(path) as f:` closes the file for you.
- Several managers can share one `with`: `with open(a) as x, open(b) as y:`.

In [ ]:
import os, tempfile
path = os.path.join(tempfile.mkdtemp(), "notes.txt")
with open(path, "w") as f:
    f.write("hello")
print(f.closed)
with open(path) as f:
    print(f.read())

**Exercise 1.** After a `with open(...)` block ends, is the file closed?

In [ ]:
import os, tempfile
path = os.path.join(tempfile.mkdtemp(), "a.txt")
with open(path, "w") as f:
    f.write("x")
is_closed = ___
check(1, is_closed)

**Exercise 2.** Write `"42"` to a file, then read it back as an `int`.

In [ ]:
import os, tempfile
path = os.path.join(tempfile.mkdtemp(), "n.txt")
with open(path, "w") as f:
    f.write("42")
with open(path) as f:
    value = int(___)
check(2, value)

**Exercise 3.** If the block raises, does cleanup still run? Write `True` or `False`.

In [ ]:
answer = ___
check(3, answer)

## 2. Writing a context manager class

- `__enter__(self)` runs at the start; what it returns is bound by `as`.
- `__exit__(self, exc_type, exc, tb)` runs at the end. The three arguments are `None` when nothing went wrong.
- Return `False` (or nothing) from `__exit__` to let an exception continue; `True` swallows it.

In [ ]:
class EventRecorder:
    def __init__(self):
        self.events = []
    def __enter__(self):
        self.events.append("start")
        return self
    def __exit__(self, exc_type, exc, tb):
        self.events.append("end")
        return False

rec = EventRecorder()
with rec:
    rec.events.append("work")
print(rec.events)

**Exercise 4.** Predict the events after the block, using the demo's `EventRecorder`.

In [ ]:
class EventRecorder:
    def __init__(self): self.events = []
    def __enter__(self):
        self.events.append("start"); return self
    def __exit__(self, *exc):
        self.events.append("end"); return False

rec = EventRecorder()
with rec:
    rec.events.append("middle")
predicted = ___
check(4, predicted)

**Exercise 5.** Make `as` give the recorder itself.

In [ ]:
class Recorder:
    def __init__(self): self.events = []
    def __enter__(self):
        return ___
    def __exit__(self, *exc):
        return False

with Recorder() as r:
    r.events.append("x")
check(5, r.events)

**Exercise 6.** The block raises. With `__exit__` returning `False`, is `"end"` still recorded, and does the error escape?

In [ ]:
class EventRecorder:
    def __init__(self): self.events = []
    def __enter__(self): return self
    def __exit__(self, *exc):
        self.events.append("end"); return False

rec = EventRecorder()
try:
    with rec:
        raise KeyError("boom")
except KeyError:
    escaped = True
predicted = (rec.events, ___)
check(6, predicted)

## 3. contextlib

- `@contextlib.contextmanager` turns a generator into a manager: code before `yield` is setup, after it is cleanup.
- Put the `yield` in `try/finally` so cleanup runs even on errors.
- `contextlib.suppress(KeyError)` ignores one kind of error for the block.

In [ ]:
from contextlib import contextmanager, suppress

@contextmanager
def tag(name):
    print(f"<{name}>")
    try:
        yield
    finally:
        print(f"</{name}>")

with tag("p"):
    print("text")
with suppress(KeyError):
    {}["missing"]
print("still running")

**Exercise 7.** Record setup and cleanup around the block.

In [ ]:
from contextlib import contextmanager
log = []

@contextmanager
def step():
    log.append("open")
    try:
        yield
    finally:
        log.append(___)

with step():
    log.append("work")
check(7, log)

**Exercise 8.** Ignore a missing key with `suppress` and keep the default.

In [ ]:
from contextlib import suppress
value = "default"
with suppress(___):
    value = {}["missing"]
check(8, value)

**Exercise 9.** Yield a value: `with temp_value() as v:` should give `99`.

In [ ]:
from contextlib import contextmanager

@contextmanager
def temp_value():
    yield ___

with temp_value() as v:
    pass
check(9, v)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### Independent build — `EventRecorder`

Implement EventRecorder(). Its events list starts empty. Entering a with block appends 'start' and binds the recorder itself. Leaving appends 'end', including on failure. Propagate the original exception.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
class EventRecorder:
    pass  # Implement the public interface in the ticket.

In [ ]:
check('EventRecorder')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** When does __exit__ run?

   - **a)** Never automatically
   - **b)** Only on success
   - **c)** When the with block ends, even on an exception

**2.** If __exit__ returns True, an exception in the block is…

   - **a)** Suppressed
   - **b)** Re-raised
   - **c)** Logged

**3.** @contextlib.contextmanager turns a ___ into a context manager.

   - **a)** class
   - **b)** dict
   - **c)** generator function with one yield

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/08', {1: 'a9c9a1f44bd6', 2: 'b0d0a46b3437', 3: '80a2b5fb6dc0'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [09 — Functional Tools, Sorting, and itertools](09_functional_tools_and_itertools.ipynb)